# 1. What you'll learn

        - compare ridge, lasso, and elastic-net coefficient paths
- choose penalty strength without using test data
- see collinearity make unregularized coefficients unstable

        **The one question this notebook answers:** How do coefficient penalties trade a little bias for stability and feature selection?

# 2. Setup

This experiment uses correlated predictors and sparse known truth, the exact conditions that make the three penalties behave differently.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

We generate 600 rows with 12 correlated features. Only features 0, 3, and 7 truly affect the target with coefficients 3, −2, and 1.5. Correlation gives the model several nearly interchangeable explanations.

**Small example:** For coefficients [4,1,0,0,0], L1 size is 5 and L2 squared size is 17. Lasso charges 5λ and can zero entries; ridge charges 17λ and tends to shrink all smoothly.

In [ ]:
rng=np.random.default_rng(SEED);n=600;p=12
latent=rng.normal(size=(n,4));mix=rng.normal(size=(4,p));X=latent@mix+rng.normal(0,.35,(n,p))
true_beta=np.zeros(p);true_beta[[0,3,7]]=[3,-2,1.5];y=X@true_beta+rng.normal(0,1,n)
features=[f"x{i}" for i in range(p)];frame=pd.DataFrame(X,columns=features);frame["target"]=y
print("Shape:",frame.shape);print("Nonzero truth:",{features[i]:true_beta[i] for i in np.flatnonzero(true_beta)});print(frame.head(2).round(2))

# 4. Explore

We inspect correlation, marginal relationships, and target scale. A feature can look predictive only because it shadows a true feature, which is why coefficient stability matters.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].imshow(frame[features].corr(), cmap="coolwarm", vmin=-1, vmax=1)
axes[0].set_xticks(range(p), features, rotation=90)
axes[0].set_yticks(range(p), features)
axes[0].grid(False)
axes[0].set_title("Correlated predictors")
axes[1].scatter(frame.x0, frame.target, s=12, alpha=.35)
axes[1].set(title="True feature x0", xlabel="x0", ylabel="target")
axes[2].hist(frame.target, bins=25, color="#176b66")
axes[2].set(title="Target scale", xlabel="target", ylabel="count")
plt.tight_layout()
plt.show()


**Takeaway:** Blocks of correlation mean several columns can substitute for one another.

**Takeaway:** A true effect is visible but still mixed with correlated contributions.

**Takeaway:** The continuous target has enough spread for RMSE differences to be meaningful.

# 5. Prepare

We split 60/20/20 into training, validation, and final test rows. Fit the scaler **only on training rows**. Fit every candidate on those same training rows; use validation RMSE to choose each family's penalty and then the family. Keep test targets out of these decisions.

Penalties act on coefficient size, so leaving one feature in dollars and another in fractions would make the penalty a unit convention rather than a modeling choice. This notebook uses one explicit holdout for selection, not inner cross-validation. If you replace it with cross-validation, put the scaler inside the estimator passed to `GridSearchCV` so every fold learns its own scaling.

In [ ]:
dev, test = train_test_split(frame, test_size=.2, random_state=SEED)
train, val = train_test_split(dev, test_size=.25, random_state=SEED)
scaler = StandardScaler().fit(train[features])
Xtr = scaler.transform(train[features])
Xv = scaler.transform(val[features])
Xte = scaler.transform(test[features])
ytr, yv, yte = (part.target.to_numpy() for part in (train, val, test))
alphas = np.logspace(-3, 1.5, 35)
print("Train / validation / test:", Xtr.shape, Xv.shape, Xte.shape)
print("Alpha range:", alphas[[0, -1]])

# 6. Train and select using validation

Ridge shares weight across correlated features; lasso can select a sparse representative; elastic net mixes both behaviors. For each alpha we record the full coefficient path. Elastic-net paths use a fixed ratio of 0.5 so the plotted change is caused by alpha alone; model selection also considers ratios 0.2 and 0.8.

1. Fit every candidate on `Xtr, ytr` only.
2. Measure each candidate on `Xv, yv` without refitting it.
3. Freeze the best settings within each family and select a family using validation RMSE.
4. Only after that decision, open the test set once for the selected model and the mean baseline.

A single validation split can give a noisy choice. The test score estimates the selected workflow's performance; it is not permission to retune and report the same test again. Also, scikit-learn normalizes the ridge and lasso losses differently, so identical alpha values do not mean identical penalty strength across families.

In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** .5

paths = {"ridge": [], "lasso": [], "elastic": []}
models, selections = {}, []
for name in paths:
    candidates = []
    ratios = [.2, .5, .8] if name == "elastic" else [None]
    for alpha in alphas:
        for ratio in ratios:
            if name == "ridge":
                model = Ridge(alpha=alpha)
            elif name == "lasso":
                model = Lasso(alpha=alpha, max_iter=5000, random_state=SEED)
            else:
                model = ElasticNet(alpha=alpha, l1_ratio=ratio,
                                   max_iter=5000, random_state=SEED)
            model.fit(Xtr, ytr)
            candidates.append((rmse(yv, model.predict(Xv)), alpha, ratio, model))
            if name != "elastic" or ratio == .5:
                paths[name].append(model.coef_.copy())
    score, alpha, ratio, model = min(candidates, key=lambda item: item[0])
    models[name] = model
    selections.append({"model": name, "alpha": alpha, "l1_ratio": ratio,
                       "validation_rmse": score,
                       "active": int(np.sum(np.abs(model.coef_) > 1e-4))})
selection_table = pd.DataFrame(selections).set_index("model")
selected_name = selection_table.validation_rmse.idxmin()
selected_model = models[selected_name]
print(selection_table.round(4))
print("Family chosen before looking at test:", selected_name)

# 7. Evaluate the frozen choice

We now compare the validation-selected model with a mean baseline on the untouched test set. The table above shows validation scores and active coefficients for all families; the final test does not choose between them.

The figure shows training coefficient paths rather than a test leaderboard. Look for smooth sharing in ridge, exact zeros in lasso, and a mix in elastic net. Sparsity is not automatically truth: correlated variables let lasso select a proxy. Save this central figure as `key_figure.png`.

In [ ]:
baseline = DummyRegressor(strategy="mean").fit(Xtr, ytr)
print(f"Final test RMSE — {selected_name}: {rmse(yte, selected_model.predict(Xte)):.3f}")
print(f"Final test RMSE — mean baseline: {rmse(yte, baseline.predict(Xte)):.3f}")
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, values) in zip(axes, paths.items()):
    ax.plot(alphas, np.asarray(values))
    ax.set_xscale("log")
    title = "elastic (L1 ratio = 0.5)" if name == "elastic" else name
    ax.set(title=title, xlabel="alpha (family-specific scale)",
           ylabel="standardized coefficient")
plt.tight_layout()
plt.savefig(Path.cwd() / "key_figure.png", dpi=160, bbox_inches="tight")
plt.show()

**Read the result:** Check whether the validation-selected model improves on the test baseline. The plotted paths explain how the penalties behave, but the lowest validation RMSE can change with the split. Ridge keeps correlated groups smooth, lasso creates zeros, and elastic net offers a compromise rather than a universally superior answer.

# 8. Break it

We repeatedly resample a nearly duplicated pair and fit OLS versus ridge. Their combined effect is identifiable, but OLS arbitrarily moves weight between copies; ridge stabilizes the individual coefficients.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
rng=np.random.default_rng(SEED);base=rng.normal(size=250);Xa=np.column_stack([base,base+rng.normal(0,.01,250)]);ya=4*base+rng.normal(0,1,250)
coef_ols=[];coef_ridge=[]
for i in range(80):
    idx=rng.choice(len(ya),len(ya),replace=True);coef_ols.append(LinearRegression().fit(Xa[idx],ya[idx]).coef_);coef_ridge.append(Ridge(alpha=5).fit(Xa[idx],ya[idx]).coef_)
coef_ols=np.asarray(coef_ols);coef_ridge=np.asarray(coef_ridge)
print("OLS coefficient SD:",coef_ols.std(0).round(3));print("Ridge coefficient SD:",coef_ridge.std(0).round(3));print("OLS/Ridge instability ratio:",round(coef_ols.std()/coef_ridge.std(),1))

**Use this when…** many predictors are correlated, sample size is limited, or stable prediction matters more than an unpenalized fit.

        **Don't use this when…** you interpret a selected lasso feature as uniquely causal or skip scaling and let measurement units determine shrinkage.

        ## Try this

        1. Set l1_ratio to .1 and .9; watch elastic-net paths move toward ridge and lasso.
2. Add 50 pure-noise features and compare test RMSE with OLS.
3. Repeat the data with uncorrelated features; lasso should recover the sparse truth more cleanly.